# Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import joblib
from sklearn.preprocessing import StandardScaler

In [2]:
# Load all raw CSVs
csv_files = glob.glob('../data/raw/*.csv')
motor_data = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
motor_data = motor_data.sort_values('Timestamp').reset_index(drop=True)

# Remove missing values and duplicates
sensor_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
motor_data = motor_data.dropna(subset=sensor_cols + ['Machine failure'])
motor_data = motor_data.drop_duplicates()

# Fit scaler and save it so app.py can use it at inference time !!! z = (x - mean) / std
scaler = StandardScaler()
motor_data[sensor_cols] = scaler.fit_transform(motor_data[sensor_cols])

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
print('Scaler saved to ../models/scaler.pkl')

# Save processed data
motor_data.to_csv('../data/processed/processed_merged_sensorData.csv', index=False)
print(f'Preprocessing complete — {len(motor_data)} rows')
print(f'Class distribution: {motor_data["Machine failure"].value_counts().to_dict()}')

Scaler saved to ../models/scaler.pkl
Preprocessing complete — 20000 rows
Class distribution: {0: 18761, 1: 1239}
